In [2]:
# ============================================
# DATA EXPLORATION - LinkedIn Job Postings Dataset
# Run this FIRST to understand the data structure
# ============================================

import pandas as pd
import numpy as np
import os

print("="*70)
print("📂 CHECKING DATASET LOCATIONS")
print("="*70)

# Check what files exist in the input directory
input_path = '/kaggle/input/'
print(f"\nContents of {input_path}:")

for root, dirs, files in os.walk(input_path):
    # Limit depth to avoid too much output
    level = root.replace(input_path, '').count(os.sep)
    if level < 3:  # Only show 3 levels deep
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        
        # Show files (limit to first 15)
        for file in files[:15]:
            file_path = os.path.join(root, file)
            try:
                size_mb = os.path.getsize(file_path) / (1024 * 1024)
                print(f"{subindent}{file} ({size_mb:.1f} MB)")
            except:
                print(f"{subindent}{file}")
        
        if len(files) > 15:
            print(f"{subindent}... and {len(files) - 15} more files")

# ============================================
# EXPLORE LINKEDIN JOB POSTINGS DATASET
# ============================================

print("\n" + "="*70)
print("📊 EXPLORING LINKEDIN JOB POSTINGS DATASET")
print("="*70)

# Find CSV files in linkedin-job-postings
linkedin_path = '/kaggle/input/linkedin-job-postings/'

if os.path.exists(linkedin_path):
    print(f"\n✅ Found linkedin-job-postings directory")
    print("\nAll files in this dataset:")
    
    all_csvs = []
    for root, dirs, files in os.walk(linkedin_path):
        for file in files:
            full_path = os.path.join(root, file)
            try:
                size_mb = os.path.getsize(full_path) / (1024 * 1024)
                print(f"  {os.path.relpath(full_path, linkedin_path)} ({size_mb:.1f} MB)")
                if file.endswith('.csv'):
                    all_csvs.append(full_path)
            except:
                print(f"  {os.path.relpath(full_path, linkedin_path)}")
    
    # Try to find the main job postings file
    possible_names = ['postings.csv', 'job_postings.csv', 'jobs.csv', 'linkedin_job_postings.csv']
    main_file = None
    
    for csv_path in all_csvs:
        filename = os.path.basename(csv_path).lower()
        if any(name in filename for name in possible_names):
            main_file = csv_path
            break
    
    if main_file is None and all_csvs:
        main_file = all_csvs[0]  # Use first CSV found
    
    if main_file:
        print(f"\n📁 Loading main file: {main_file}")
        
        # Load with nrows first to check structure
        df_sample = pd.read_csv(main_file, nrows=5)
        print(f"\n✅ Successfully loaded sample")
        
        # Get full row count
        print("\n🔄 Counting total rows (this may take a moment)...")
        total_rows = sum(1 for _ in open(main_file)) - 1  # -1 for header
        print(f"Total rows: {total_rows:,}")
        
        print(f"\n" + "-"*60)
        print("COLUMN INFORMATION")
        print("-"*60)
        print(f"Number of columns: {len(df_sample.columns)}")
        print(f"\nColumn names:")
        for i, col in enumerate(df_sample.columns):
            print(f"  {i+1}. {col}")
        
        # Load more rows for detailed analysis
        print("\n🔄 Loading 1000 rows for detailed analysis...")
        df = pd.read_csv(main_file, nrows=1000)
        
        print(f"\n" + "-"*60)
        print("DATA TYPES")
        print("-"*60)
        print(df.dtypes)
        
        print(f"\n" + "-"*60)
        print("MISSING VALUES (in first 1000 rows)")
        print("-"*60)
        missing = df.isnull().sum()
        missing_pct = (missing / len(df) * 100).round(1)
        missing_df = pd.DataFrame({'missing': missing, 'percent': missing_pct})
        print(missing_df[missing_df['missing'] > 0].sort_values('missing', ascending=False))
        
        print(f"\n" + "-"*60)
        print("SAMPLE DATA (First 3 rows)")
        print("-"*60)
        pd.set_option('display.max_colwidth', 80)
        pd.set_option('display.max_columns', None)
        print(df.head(3).T)  # Transpose for better readability
        
        print(f"\n" + "-"*60)
        print("KEY COLUMNS ANALYSIS")
        print("-"*60)
        
        # Check for common column names
        key_cols = {
            'title': ['title', 'job_title', 'position', 'role'],
            'company': ['company', 'company_name', 'employer', 'company_id'],
            'description': ['description', 'job_description', 'desc', 'summary'],
            'location': ['location', 'job_location', 'city', 'state'],
            'salary': ['salary', 'min_salary', 'max_salary', 'med_salary', 'pay'],
            'skills': ['skills', 'required_skills', 'skill'],
            'job_id': ['job_id', 'id', 'posting_id', 'job_link']
        }
        
        for category, possible_cols in key_cols.items():
            found = [col for col in df.columns if any(p in col.lower() for p in possible_cols)]
            if found:
                print(f"\n📌 {category.upper()} columns found: {found}")
                for col in found[:2]:  # Show details for first 2 matches
                    print(f"   '{col}':")
                    print(f"   - Non-null: {df[col].notna().sum()}/{len(df)}")
                    print(f"   - Unique: {df[col].nunique()}")
                    if df[col].dtype == 'object':
                        sample = df[col].dropna().iloc[0] if df[col].notna().any() else "N/A"
                        sample_str = str(sample)[:100] + "..." if len(str(sample)) > 100 else str(sample)
                        print(f"   - Sample: {sample_str}")
                    else:
                        print(f"   - Range: {df[col].min()} to {df[col].max()}")
else:
    print(f"❌ linkedin-job-postings directory not found")

# ============================================
# EXPLORE SKILLS DATASET (if available)
# ============================================

print("\n" + "="*70)
print("📊 EXPLORING SKILLS DATASET (1.3M LinkedIn Jobs)")
print("="*70)

skills_path = '/kaggle/input/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024'

if os.path.exists(skills_path):
    print(f"\n✅ Found skills dataset directory")
    print("\nFiles in this dataset:")
    
    for root, dirs, files in os.walk(skills_path):
        for file in files:
            full_path = os.path.join(root, file)
            try:
                size_mb = os.path.getsize(full_path) / (1024 * 1024)
                print(f"  {file} ({size_mb:.1f} MB)")
            except:
                print(f"  {file}")
    
    # Try to load job_skills.csv
    skills_file = os.path.join(skills_path, 'job_skills.csv')
    if os.path.exists(skills_file):
        print(f"\n📁 Loading job_skills.csv (sample)...")
        skills_df = pd.read_csv(skills_file, nrows=1000)
        
        print(f"\n" + "-"*60)
        print("JOB_SKILLS.CSV STRUCTURE")
        print("-"*60)
        print(f"Columns: {skills_df.columns.tolist()}")
        print(f"\nData types:")
        print(skills_df.dtypes)
        print(f"\nSample rows:")
        print(skills_df.head(3))
        
        # Check skills column
        skills_col = None
        for col in skills_df.columns:
            if 'skill' in col.lower():
                skills_col = col
                break
        
        if skills_col:
            print(f"\n📌 Skills column: '{skills_col}'")
            print(f"Sample skills:")
            for i, val in enumerate(skills_df[skills_col].dropna().head(5)):
                print(f"  {i+1}. {str(val)[:150]}...")
    
    # Try to load linkedin_job_postings.csv
    jobs_file = os.path.join(skills_path, 'linkedin_job_postings.csv')
    if os.path.exists(jobs_file):
        print(f"\n📁 Loading linkedin_job_postings.csv (sample)...")
        jobs2_df = pd.read_csv(jobs_file, nrows=1000)
        
        print(f"\n" + "-"*60)
        print("LINKEDIN_JOB_POSTINGS.CSV STRUCTURE")
        print("-"*60)
        print(f"Columns: {jobs2_df.columns.tolist()}")
        print(f"\nData types:")
        print(jobs2_df.dtypes)
        print(f"\nSample rows:")
        print(jobs2_df.head(3))
else:
    print(f"❌ Skills dataset directory not found")
    print("Note: Make sure you added '1.3M LinkedIn Jobs and Skills 2024' dataset")

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*70)
print("✅ EXPLORATION COMPLETE")
print("="*70)
print("\n👆 Copy ALL the output above and share it with me!")
print("\nI need to see:")
print("  1. Exact file paths")
print("  2. Column names for job postings")
print("  3. Column names for skills data")
print("  4. Sample data structure")

📂 CHECKING DATASET LOCATIONS

Contents of /kaggle/input/:
/
datasets/
  asaniczka/
    1-3m-linkedin-jobs-and-skills-2024/
      job_summary.csv (4865.7 MB)
      job_skills.csv (641.6 MB)
      linkedin_job_postings.csv (396.1 MB)
linkedin-job-postings/
  job_postings.csv (61.1 MB)
  company_details/
    companies.csv (5.6 MB)
    company_industries.csv (0.4 MB)
    company_specialities.csv (2.9 MB)
    employee_counts.csv (0.5 MB)
  job_details/
    benefits.csv (0.4 MB)
    job_skills.csv (0.4 MB)
    job_industries.csv (0.3 MB)

📊 EXPLORING LINKEDIN JOB POSTINGS DATASET

✅ Found linkedin-job-postings directory

All files in this dataset:
  job_postings.csv (61.1 MB)
  company_details/companies.csv (5.6 MB)
  company_details/company_industries.csv (0.4 MB)
  company_details/company_specialities.csv (2.9 MB)
  company_details/employee_counts.csv (0.5 MB)
  job_details/benefits.csv (0.4 MB)
  job_details/job_skills.csv (0.4 MB)
  job_details/job_industries.csv (0.3 MB)

📁 Loading main

In [4]:
# ============================================
# NOTEBOOK 2: Job Embeddings & FAISS Index
# ============================================
# Dataset: LinkedIn Job Postings (254K jobs)
# Files:
#   - /kaggle/input/linkedin-job-postings/job_postings.csv
#   - /kaggle/input/linkedin-job-postings/job_details/job_skills.csv
#   - /kaggle/input/linkedin-job-postings/company_details/companies.csv
# ============================================

import pandas as pd
import numpy as np
import json
import pickle
import re
import os
import subprocess

# ============================================
# SETUP: Install required packages
# ============================================

print("📦 Installing packages...")
subprocess.run(['pip', 'install', 'sentence-transformers', 'faiss-cpu', '-q'], 
               check=True, capture_output=True)

from sentence_transformers import SentenceTransformer
import faiss

print("✅ Packages installed and imported")

# ============================================
# PART 1: Load Job Postings Data
# ============================================

print("\n" + "="*60)
print("PART 1: Loading Job Postings Data")
print("="*60)

# Load main job postings
jobs_df = pd.read_csv('/kaggle/input/linkedin-job-postings/job_postings.csv')
print(f"✅ Loaded {len(jobs_df):,} job postings")
print(f"Columns: {jobs_df.columns.tolist()}")

# Rename job description column to avoid conflict with company description
if 'description' in jobs_df.columns:
    jobs_df = jobs_df.rename(columns={'description': 'job_description'})
    print("✅ Renamed 'description' to 'job_description'")

# ============================================
# PART 2: Load and Join Skills Data
# ============================================

print("\n" + "="*60)
print("PART 2: Loading Skills Data")
print("="*60)

skills_path = '/kaggle/input/linkedin-job-postings/job_details/job_skills.csv'

if os.path.exists(skills_path):
    skills_df = pd.read_csv(skills_path)
    print(f"✅ Loaded skills file: {len(skills_df):,} entries")
    print(f"Columns: {skills_df.columns.tolist()}")
    
    # Check if job_id exists for joining
    if 'job_id' in skills_df.columns:
        # Find the skill column
        if 'skill_abr' in skills_df.columns:
            skill_col = 'skill_abr'
        elif 'skill' in skills_df.columns:
            skill_col = 'skill'
        else:
            skill_col = skills_df.columns[1]
        
        print(f"Using skill column: '{skill_col}'")
        
        # Group skills by job
        job_skills_grouped = skills_df.groupby('job_id')[skill_col].apply(list).reset_index()
        job_skills_grouped.columns = ['job_id', 'skills_list']
        
        # Merge with jobs
        jobs_df = jobs_df.merge(job_skills_grouped, on='job_id', how='left')
        print(f"✅ Merged skills with jobs")
        print(f"Jobs with skills: {jobs_df['skills_list'].notna().sum():,}")
    else:
        jobs_df['skills_list'] = None
else:
    print(f"⚠️ Skills file not found")
    jobs_df['skills_list'] = None

# ============================================
# PART 3: Load Company Data
# ============================================

print("\n" + "="*60)
print("PART 3: Loading Company Data")
print("="*60)

companies_path = '/kaggle/input/linkedin-job-postings/company_details/companies.csv'

if os.path.exists(companies_path):
    companies_df = pd.read_csv(companies_path)
    print(f"✅ Loaded {len(companies_df):,} companies")
    print(f"Columns: {companies_df.columns.tolist()}")
    
    # Only keep company_id and name (avoid description conflict)
    companies_df = companies_df[['company_id', 'name']].copy()
    companies_df = companies_df.rename(columns={'name': 'company_name'})
    
    # Merge with jobs
    jobs_df = jobs_df.merge(companies_df, on='company_id', how='left')
    print(f"✅ Merged company names with jobs")
else:
    print(f"⚠️ Companies file not found")
    jobs_df['company_name'] = None

# Check columns after merging
print(f"\nColumns after merging: {jobs_df.columns.tolist()}")

# ============================================
# PART 4: Clean and Prepare Data
# ============================================

print("\n" + "="*60)
print("PART 4: Cleaning and Preparing Data")
print("="*60)

# Fill missing values
jobs_df['title'] = jobs_df['title'].fillna('Unknown Position')
jobs_df['job_description'] = jobs_df['job_description'].fillna('')
jobs_df['location'] = jobs_df['location'].fillna('Unknown Location')
jobs_df['company_name'] = jobs_df['company_name'].fillna('Unknown Company')

# Clean description text
def clean_text(text):
    """Clean job description text"""
    if pd.isna(text) or text == '':
        return ''
    
    text = str(text)
    # Remove HTML tags if any
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s\.\,\-\(\)\:\;\/\&]', ' ', text)
    # Trim
    text = text.strip()
    
    return text

print("🔄 Cleaning descriptions...")
jobs_df['description_clean'] = jobs_df['job_description'].apply(clean_text)

# Filter out jobs with empty descriptions
original_count = len(jobs_df)
jobs_df = jobs_df[jobs_df['description_clean'].str.len() > 50].reset_index(drop=True)
print(f"✅ Filtered jobs with descriptions: {len(jobs_df):,} (removed {original_count - len(jobs_df):,})")

# ============================================
# PART 5: Sample Jobs for Embedding
# ============================================

print("\n" + "="*60)
print("PART 5: Sampling Jobs")
print("="*60)

# Sample 15,000 jobs if we have more
SAMPLE_SIZE = 15000
np.random.seed(42)

if len(jobs_df) > SAMPLE_SIZE:
    jobs_df = jobs_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"✅ Sampled to {len(jobs_df):,} jobs")
else:
    print(f"✅ Using all {len(jobs_df):,} jobs")

# Show sample
print(f"\nSample job titles:")
for title in jobs_df['title'].head(10):
    print(f"  - {title}")

# ============================================
# PART 6: Extract Skills from Descriptions
# ============================================

print("\n" + "="*60)
print("PART 6: Extracting Skills from Descriptions")
print("="*60)

# Comprehensive skill patterns
SKILL_PATTERNS = [
    # Programming Languages
    'python', 'java', 'javascript', 'c\\+\\+', 'c#', 'ruby', 'php', 'swift', 
    'kotlin', 'scala', 'go', 'rust', 'typescript', 'sql', 'html', 'css',
    'perl', 'matlab', 'r programming', 'bash', 'shell',
    
    # Frameworks
    'react', 'angular', 'vue', 'django', 'flask', 'spring', 'node\\.js',
    'express', 'asp\\.net', 'laravel', 'rails', 'next\\.js', 'fastapi',
    
    # Data Science & ML
    'tensorflow', 'pytorch', 'keras', 'scikit-learn', 'pandas', 'numpy',
    'machine learning', 'deep learning', 'neural network', 'nlp',
    'computer vision', 'data analysis', 'data science', 'statistics',
    'predictive modeling', 'data mining', 'big data',
    
    # Databases
    'mysql', 'postgresql', 'mongodb', 'redis', 'oracle', 'sql server',
    'dynamodb', 'elasticsearch', 'cassandra', 'sqlite',
    
    # Cloud & DevOps
    'aws', 'azure', 'gcp', 'google cloud', 'docker', 'kubernetes',
    'jenkins', 'terraform', 'ansible', 'ci/cd', 'git', 'github',
    'linux', 'devops', 'microservices', 'rest api',
    
    # Tools
    'excel', 'power bi', 'tableau', 'jira', 'confluence', 'slack',
    'salesforce', 'sap', 'quickbooks', 'adobe',
    
    # Methodologies
    'agile', 'scrum', 'kanban', 'waterfall', 'six sigma', 'lean',
    
    # Soft Skills
    'communication', 'leadership', 'teamwork', 'problem solving',
    'project management', 'time management', 'presentation',
    
    # Domain specific
    'accounting', 'finance', 'marketing', 'sales', 'customer service',
    'healthcare', 'nursing', 'legal', 'compliance', 'engineering',
    'manufacturing', 'supply chain', 'logistics', 'operations'
]

def extract_skills_from_text(text):
    """Extract skills from job description"""
    if pd.isna(text) or text == '':
        return []
    
    text_lower = text.lower()
    found_skills = []
    
    for skill in SKILL_PATTERNS:
        pattern = r'\b' + skill + r'\b'
        if re.search(pattern, text_lower):
            clean_skill = skill.replace('\\', '')
            found_skills.append(clean_skill)
    
    return list(set(found_skills))

# Extract skills from descriptions
print("🔄 Extracting skills from job descriptions...")
jobs_df['extracted_skills'] = jobs_df['description_clean'].apply(extract_skills_from_text)

# Combine with existing skills if available
def combine_skills(row):
    """Combine extracted skills with any existing skills"""
    skills = set()
    
    # Add extracted skills
    if isinstance(row['extracted_skills'], list):
        skills.update([s.lower() for s in row['extracted_skills']])
    
    # Add skills from skills_list column if available
    if isinstance(row.get('skills_list'), list):
        skills.update([str(s).lower() for s in row['skills_list'] if s])
    
    return list(skills)

jobs_df['all_skills'] = jobs_df.apply(combine_skills, axis=1)
jobs_df['skill_count'] = jobs_df['all_skills'].apply(len)

print(f"✅ Skills extracted")
print(f"Jobs with skills: {(jobs_df['skill_count'] > 0).sum():,}")
print(f"Average skills per job: {jobs_df['skill_count'].mean():.1f}")

# ============================================
# PART 7: Build Skill-to-Job Mapping
# ============================================

print("\n" + "="*60)
print("PART 7: Building Skill-to-Job Mapping")
print("="*60)

from collections import defaultdict

skill_stats = defaultdict(lambda: {'count': 0, 'job_titles': set()})

for idx, row in jobs_df.iterrows():
    for skill in row['all_skills']:
        skill_stats[skill]['count'] += 1
        skill_stats[skill]['job_titles'].add(row['title'])

# Convert to serializable format
skill_job_mapping = {}
for skill, stats in skill_stats.items():
    skill_job_mapping[skill] = {
        'count': stats['count'],
        'job_titles': list(stats['job_titles'])[:20]
    }

# Sort by count
skill_job_mapping = dict(sorted(
    skill_job_mapping.items(),
    key=lambda x: x[1]['count'],
    reverse=True
))

print(f"✅ Mapped {len(skill_job_mapping)} unique skills")
print(f"\nTop 15 skills:")
for i, (skill, data) in enumerate(list(skill_job_mapping.items())[:15]):
    print(f"  {i+1}. {skill}: {data['count']} jobs")

# Save skill mapping
with open('skill_job_mapping.json', 'w') as f:
    json.dump(skill_job_mapping, f, indent=2)

print("\n✅ Saved: skill_job_mapping.json")

# ============================================
# PART 8: Create Combined Text for Embedding
# ============================================

print("\n" + "="*60)
print("PART 8: Preparing Text for Embedding")
print("="*60)

def create_embedding_text(row):
    """Create combined text for embedding"""
    parts = []
    
    # Title (most important)
    parts.append(str(row['title']))
    
    # Company
    if row['company_name'] and row['company_name'] != 'Unknown Company':
        parts.append(f"at {row['company_name']}")
    
    # Location
    if row['location'] and row['location'] != 'Unknown Location':
        parts.append(f"in {row['location']}")
    
    # Skills
    if row['all_skills']:
        parts.append(f"Skills: {', '.join(row['all_skills'][:15])}")
    
    # Description (truncated)
    desc = str(row['description_clean'])
    desc = desc[:1500] if len(desc) > 1500 else desc
    parts.append(desc)
    
    return ' | '.join(parts)

jobs_df['embedding_text'] = jobs_df.apply(create_embedding_text, axis=1)

# Truncate to reasonable length for embedding
jobs_df['embedding_text'] = jobs_df['embedding_text'].str[:2048]

print(f"✅ Created embedding text for {len(jobs_df)} jobs")
print(f"\nSample embedding text (first 300 chars):")
print(jobs_df['embedding_text'].iloc[0][:300])

# ============================================
# PART 9: Generate Embeddings
# ============================================

print("\n" + "="*60)
print("PART 9: Generating Embeddings")
print("="*60)

# Load sentence transformer model
print("🔄 Loading SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = model.get_sentence_embedding_dimension()
print(f"✅ Model loaded: {embedding_dim} dimensions")

# Generate embeddings in batches
print(f"\n🔄 Generating embeddings for {len(jobs_df)} jobs...")
batch_size = 64
embeddings_list = []

for i in range(0, len(jobs_df), batch_size):
    batch_texts = jobs_df['embedding_text'].iloc[i:i+batch_size].tolist()
    batch_embeddings = model.encode(batch_texts, show_progress_bar=False)
    embeddings_list.append(batch_embeddings)
    
    # Progress update every 50 batches
    if (i // batch_size) % 50 == 0:
        print(f"  Processed {min(i + batch_size, len(jobs_df)):,}/{len(jobs_df):,} jobs...")

# Stack all embeddings
embeddings = np.vstack(embeddings_list).astype('float32')
print(f"\n✅ Generated embeddings: {embeddings.shape}")

# Save embeddings
np.save('job_embeddings.npy', embeddings)
print("✅ Saved: job_embeddings.npy")

# ============================================
# PART 10: Build FAISS Index
# ============================================

print("\n" + "="*60)
print("PART 10: Building FAISS Index")
print("="*60)

# Normalize embeddings for cosine similarity
embeddings_normalized = embeddings.copy()
faiss.normalize_L2(embeddings_normalized)

# Create FAISS index (Inner Product = cosine similarity for normalized vectors)
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings_normalized)

print(f"✅ FAISS index built: {index.ntotal} vectors, {embedding_dim} dimensions")

# Save index
faiss.write_index(index, 'job_index.faiss')
print("✅ Saved: job_index.faiss")

# ============================================
# PART 11: Save Job Metadata
# ============================================

print("\n" + "="*60)
print("PART 11: Saving Job Metadata")
print("="*60)

# Prepare metadata for each job
job_metadata = []

for idx, row in jobs_df.iterrows():
    meta = {
        'job_id': int(row['job_id']) if pd.notna(row['job_id']) else None,
        'title': str(row['title']) if pd.notna(row['title']) else 'Unknown',
        'company_name': str(row['company_name']) if pd.notna(row['company_name']) else 'Unknown',
        'location': str(row['location']) if pd.notna(row['location']) else 'Unknown',
        'description': str(row['description_clean'])[:500] if pd.notna(row['description_clean']) else '',
        'skills': row['all_skills'] if isinstance(row['all_skills'], list) else [],
        'work_type': str(row['formatted_work_type']) if pd.notna(row.get('formatted_work_type')) else None,
        'experience_level': str(row['formatted_experience_level']) if pd.notna(row.get('formatted_experience_level')) else None,
        'min_salary': float(row['min_salary']) if pd.notna(row.get('min_salary')) else None,
        'max_salary': float(row['max_salary']) if pd.notna(row.get('max_salary')) else None,
        'pay_period': str(row['pay_period']) if pd.notna(row.get('pay_period')) else None,
    }
    job_metadata.append(meta)

# Save metadata
with open('job_metadata.pkl', 'wb') as f:
    pickle.dump(job_metadata, f)

print(f"✅ Saved metadata for {len(job_metadata)} jobs")

# Also save sample as JSON for inspection
with open('job_metadata_sample.json', 'w') as f:
    json.dump(job_metadata[:10], f, indent=2)
print("✅ Saved: job_metadata_sample.json (first 10 jobs)")

# ============================================
# PART 12: Test the Search System
# ============================================

print("\n" + "="*60)
print("PART 12: Testing Search System")
print("="*60)

def search_jobs(query, k=5):
    """Search for similar jobs given a query"""
    # Encode query
    query_embedding = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)
    
    # Search
    distances, indices = index.search(query_embedding, k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        job = job_metadata[idx]
        results.append({
            'score': float(dist),
            'title': job.get('title'),
            'company': job.get('company_name'),
            'location': job.get('location'),
            'skills': job.get('skills', [])[:5]
        })
    
    return results

# Test queries
test_queries = [
    "Python developer with machine learning experience",
    "Marketing manager with social media skills",
    "Registered nurse in healthcare"
]

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    print("-" * 50)
    
    results = search_jobs(query, k=3)
    for i, result in enumerate(results):
        print(f"\n  {i+1}. {result['title']}")
        print(f"     Company: {result['company']}")
        print(f"     Location: {result['location']}")
        print(f"     Score: {result['score']:.3f}")
        if result['skills']:
            print(f"     Skills: {', '.join(result['skills'])}")

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*60)
print("✅ NOTEBOOK 2 COMPLETE")
print("="*60)

print("\n📁 Generated Files:")
print("   1. job_embeddings.npy - Job embeddings (384-dim vectors)")
print("   2. job_index.faiss - FAISS index for similarity search")
print("   3. job_metadata.pkl - Job details (title, company, skills, etc.)")
print("   4. skill_job_mapping.json - Skill → job titles mapping")
print("   5. job_metadata_sample.json - Sample metadata for inspection")

print("\n📊 Statistics:")
print(f"   Total jobs indexed: {len(jobs_df):,}")
print(f"   Embedding dimensions: {embedding_dim}")
print(f"   Unique skills mapped: {len(skill_job_mapping)}")
print(f"   Jobs with skills: {(jobs_df['skill_count'] > 0).sum():,}")
print(f"   Average skills per job: {jobs_df['skill_count'].mean():.1f}")

print("\n📥 Download all 5 files and place in 'trained_models/' folder")
print("="*60)

📦 Installing packages...
✅ Packages installed and imported

PART 1: Loading Job Postings Data
✅ Loaded 15,886 job postings
Columns: ['job_id', 'company_id', 'title', 'description', 'max_salary', 'med_salary', 'min_salary', 'pay_period', 'formatted_work_type', 'location', 'applies', 'original_listed_time', 'remote_allowed', 'views', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type']
✅ Renamed 'description' to 'job_description'

PART 2: Loading Skills Data
✅ Loaded skills file: 27,899 entries
Columns: ['job_id', 'skill_abr']
Using skill column: 'skill_abr'
✅ Merged skills with jobs
Jobs with skills: 15,249

PART 3: Loading Company Data
✅ Loaded 6,063 companies
Columns: ['company_id', 'name', 'description', 'company_size', 'state', 'country', 'city', 'zip_code', 'address', 'url']
✅ Merged company names with jobs

Columns a

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded: 384 dimensions

🔄 Generating embeddings for 15000 jobs...
  Processed 64/15,000 jobs...
  Processed 3,264/15,000 jobs...
  Processed 6,464/15,000 jobs...
  Processed 9,664/15,000 jobs...
  Processed 12,864/15,000 jobs...

✅ Generated embeddings: (15000, 384)
✅ Saved: job_embeddings.npy

PART 10: Building FAISS Index
✅ FAISS index built: 15000 vectors, 384 dimensions
✅ Saved: job_index.faiss

PART 11: Saving Job Metadata
✅ Saved metadata for 15000 jobs
✅ Saved: job_metadata_sample.json (first 10 jobs)

PART 12: Testing Search System

🔍 Query: 'Python developer with machine learning experience'
--------------------------------------------------

  1. Python Developer
     Company: Diamondpick
     Location: Plano, TX
     Score: 0.662
     Skills: java, jira, flask, django, eng

  2. Environmental Engineer & Data Scientist
     Company: Backpack Talent
     Location: Denver, CO
     Score: 0.615
     Skills: machine learning, cnsl, sci, eng, data science

  3. Devops Engi